<a href="https://colab.research.google.com/github/mducdaf2/jetracer-car/blob/dev/notebooks/TrafficSignModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

# 1. Lấy token từ khóa bí mật 'kaggle'
os.environ['KAGGLE_API_TOKEN'] = userdata.get('kaggle')

In [2]:
!kaggle --version
!kaggle config view

Kaggle CLI 2.0.2
Configuration values from /root/.config/kaggle
- username: daf2pro
- auth_method: ACCESS_TOKEN
- path: None
- proxy: None
- competition: None


In [3]:
!rm -f jetracer-smartcity.zip
!rm -rf dataset/

In [4]:
!kaggle datasets download -d daf2pro/jetracer-smartcity --unzip -p ./dataset

Dataset URL: https://www.kaggle.com/datasets/daf2pro/jetracer-smartcity
License(s): CC0-1.0
100% 39.1M/39.1M [00:03<00:00, 13.3MB/s]



In [5]:
os.listdir("dataset")

['valid', 'README.dataset.txt', 'README.roboflow.txt', 'train', 'test']

In [6]:
# Cài đặt Ultralytics và các thư viện cần thiết
!pip install ultralytics pycocotools

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 6.8 MB/s eta 0:00:00


In [11]:
import os
import yaml
import shutil
import glob
from ultralytics import YOLO
from ultralytics.data.converter import convert_coco

In [20]:
import os
import json
import shutil
import glob

def convert_roboflow_coco_to_yolo(json_path, output_labels_dir):
    os.makedirs(output_labels_dir, exist_ok=True)

    with open(json_path, 'r', encoding='utf-8') as f:
        coco = json.load(f)

    # Lấy các categories có id > 0 và map về index 0 -> 5
    valid_cats = [cat for cat in coco['categories'] if cat['id'] > 0]
    cat_id_to_yolo_id = {cat['id']: idx for idx, cat in enumerate(sorted(valid_cats, key=lambda x: x['id']))}

    image_dict = {img['id']: img for img in coco['images']}
    annotations_per_image = {img['id']: [] for img in coco['images']}

    for ann in coco['annotations']:
        if ann['category_id'] in cat_id_to_yolo_id:
            annotations_per_image[ann['image_id']].append(ann)

    for img_id, img_info in image_dict.items():
        img_w, img_h = img_info['width'], img_info['height']
        image_filename = os.path.basename(img_info['file_name'])
        txt_filename = os.path.splitext(image_filename)[0] + '.txt'
        txt_path = os.path.join(output_labels_dir, txt_filename)

        lines = []
        for ann in annotations_per_image[img_id]:
            yolo_cat_id = cat_id_to_yolo_id[ann['category_id']]
            x, y, w, h = ann['bbox']

            x_center = (x + w / 2.0) / img_w
            y_center = (y + h / 2.0) / img_h
            w_norm = w / img_w
            h_norm = h / img_h

            lines.append(f"{yolo_cat_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

        with open(txt_path, 'w') as f:
            f.write('\n'.join(lines))

# 1. Làm sạch folder cũ
!rm -rf /content/dataset_yolo

# 2. Convert nhãn & copy ảnh cho cả 3 tập train, val, test
splits = ['train', 'valid', 'test']

for split in splits:
    json_file = f'dataset/{split}/_annotations.coco.json'
    if os.path.exists(json_file):
        # Tạo nhãn
        convert_roboflow_coco_to_yolo(json_file, f'/content/dataset_yolo/{split}/labels')

        # Copy ảnh
        os.makedirs(f'/content/dataset_yolo/{split}/images', exist_ok=True)
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
            for img_path in glob.glob(f'dataset/{split}/{ext}'):
                shutil.copy(img_path, f'/content/dataset_yolo/{split}/images/')

print("✅ Đã xử lý xong dữ liệu cho cả train, val và test!")

✅ Đã xử lý xong dữ liệu cho cả train, val và test!


In [21]:
import yaml

data_yaml = {
    'path': '/content/dataset_yolo',
    'train': 'train/images',
    'val': 'valid/images',     # Đã trỏ chính xác về val/images
    'test': 'test/images',
    'names': {
        0: "green-light",
        1: "left-turn-sign",
        2: "prohibition sign",
        3: "red-light",
        4: "right-turn-sign",
        5: "straight-ahead-sign"
    }
}

with open('data.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print("✅ Đã cập nhật file data.yaml!")

✅ Đã cập nhật file data.yaml!


In [22]:
# Tải mô hình YOLOv8 Nano
model = YOLO('yolov8n.pt')

# Tiến hành huấn luyện với các tham số tối ưu
results = model.train(
    data='data.yaml',
    epochs=100,         # Tăng epoch lên để mô hình hội tụ tốt hơn (có thể stop sớm bằng patience)
    patience=15,        # Tự động dừng nếu 15 epoch liên tiếp không cải thiện (tiết kiệm thời gian)
    imgsz=416,          # Tối ưu: Giảm từ 640 xuống 416 giúp Jetson Nano tăng gấp đôi FPS
    batch=16,           # Tốt trên GPU Colab
    workers=4,          # Giảm worker để tránh nghẽn RAM/CPU
    device=0,

    # Data Augmentation (Tăng cường dữ liệu cho xe tự hành/nhận diện biển báo)
    degrees=10.0,       # Xoay nhẹ ảnh (phù hợp khi xe nghiêng/góc quay lệch)
    scale=0.5,          # Co giãn kích thước (giúp nhận diện biển báo từ xa/gần)
    fliplr=0.0,         # QUAN TRỌNG: Tắt lật ngang vì biển rẽ trái/phải bị lật sẽ sai bản chất!
    mosaic=1.0          # Giữ mosaic để học vật thể nhỏ tốt hơn
)

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=

In [23]:
import glob
from PIL import Image

# Lấy danh sách ảnh trong tập train
image_paths = glob.glob('dataset/train/*.jpg') + glob.glob('dataset/train/*.png')

print(f"Tổng số ảnh tìm thấy: {len(image_paths)}")
print("-" * 30)

# In kích thước của 10 ảnh đầu tiên
for img_path in image_paths[:10]:
    with Image.open(img_path) as img:
        width, height = img.size
        print(f"Tên file: {img_path.split('/')[-1]} | Kích thước (W x H): {width}x{height}")

Tổng số ảnh tìm thấy: 384
------------------------------
Tên file: img_1786612704923_jpg.rf.ywmkpvKWD2JYhGvWVrGO.jpg | Kích thước (W x H): 640x480
Tên file: img_1786620719912_jpg.rf.bMxfUmDbEX58COWvjVVZ.jpg | Kích thước (W x H): 640x480
Tên file: img_1786620647889_jpg.rf.XiRIM32HaWWDEwwDU5kF.jpg | Kích thước (W x H): 640x480
Tên file: img_1786620903806_jpg.rf.1ZK92UgYeE8vNMISMk5a.jpg | Kích thước (W x H): 640x480
Tên file: img_1786612695424_jpg.rf.XWrN8CGlJ1HCaC7zIWSg.jpg | Kích thước (W x H): 640x480
Tên file: img_1786621181595_jpg.rf.WlPnuvqwVrDjo8wp9q0B.jpg | Kích thước (W x H): 640x480
Tên file: img_1786621176822_jpg.rf.LDzsnpIOBwzBknvr5O3G.jpg | Kích thước (W x H): 640x480
Tên file: img_1786612707417_jpg.rf.3b37E243nBBRDnNn8M9k.jpg | Kích thước (W x H): 640x480
Tên file: img_1786612686453_jpg.rf.YZDITKb9UeGcOL1iSshw.jpg | Kích thước (W x H): 640x480
Tên file: img_1786612708925_jpg.rf.5dBxw0FMWquOQllYGUVN.jpg | Kích thước (W x H): 640x480


In [24]:
from ultralytics import YOLO

# Load file weights từ đợt train vừa rồi
model = YOLO('/content/runs/detect/train-2/weights/best.pt')

# Export ONNX chuẩn kích thước (Height=480, Width=640)
model.export(format='onnx', imgsz=[480, 640], simplify=True)

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 6.1 GFLOPs

PyTorch: starting from '/content/runs/detect/train-2/weights/best.pt' with input shape (1, 3, 480, 640) BCHW and output shape(s) (1, 10, 6300) (5.9 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 210ms
Prepared 4 packages in 1.84s
Installed 4 packages in 272ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.29.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 3.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming w

'/content/runs/detect/train-2/weights/best.onnx'